<a href="https://colab.research.google.com/github/antoniocesar-amf/ppcompann/blob/main/Dataset_IMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup
!cat aclImdb/train/pos/4077_10.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  2693k      0  0:00:30  0:00:30 --:--:-- 5211k
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drin

In [2]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"

for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [3]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
!pip install transformers

In [5]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
results = pipe(["This is a great movie", "This is a bad movie"])
for result in results:
  print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9957
label: NEGATIVE, with score: 0.995


In [7]:
from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
for batch, (texts, targets) in enumerate(test_ds):
  for i in range(len(texts)):
    current_text = texts[i].numpy().decode('utf-8')
    yp = pipe(current_text, truncation=True, max_length=512)

    true_label = int(targets[i].numpy() == 1)
    test_labels.append(true_label)

    predicted_label_is_positive = (yp[0]['label'] == 'POSITIVE')
    y_pred.append(predicted_label_is_positive)

  print(f"{batch} {time.time()-start:.2f} s - Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("Final Accuracy", accuracy_score(test_labels, y_pred))

0 12.67 s - Accuracy: 0.9375
1 29.58 s - Accuracy: 0.9219
2 43.43 s - Accuracy: 0.9167
3 57.05 s - Accuracy: 0.9219
4 79.70 s - Accuracy: 0.9062
5 95.44 s - Accuracy: 0.9115
6 109.72 s - Accuracy: 0.9196
7 123.07 s - Accuracy: 0.9258
8 140.71 s - Accuracy: 0.9340
9 155.30 s - Accuracy: 0.9250
10 170.78 s - Accuracy: 0.9261
11 190.33 s - Accuracy: 0.9219
12 206.28 s - Accuracy: 0.9279
13 222.63 s - Accuracy: 0.9219
14 237.57 s - Accuracy: 0.9250
15 254.11 s - Accuracy: 0.9258
16 266.77 s - Accuracy: 0.9246
17 283.05 s - Accuracy: 0.9271
18 300.29 s - Accuracy: 0.9276
19 315.63 s - Accuracy: 0.9297
20 331.14 s - Accuracy: 0.9241
21 344.64 s - Accuracy: 0.9247
22 359.69 s - Accuracy: 0.9266
23 373.90 s - Accuracy: 0.9284
24 387.51 s - Accuracy: 0.9300
25 402.68 s - Accuracy: 0.9303
26 417.51 s - Accuracy: 0.9294
27 431.06 s - Accuracy: 0.9275
28 447.25 s - Accuracy: 0.9267
29 461.51 s - Accuracy: 0.9260
30 477.49 s - Accuracy: 0.9264
31 491.05 s - Accuracy: 0.9277
32 506.31 s - Accuracy: 